<a href="https://colab.research.google.com/github/panupong2548/algorithm-Perceptron/blob/main/PerceptronPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PerceptronPytorch


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_classification

class PerceptronGDPyTorch:
    """Perceptron classifier with Batch Gradient Descent using PyTorch."""
    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X, y):
        # 1. แปลงข้อมูล X และ y ให้อยู่ในรูปของ PyTorch Tensors
        # ตั้งค่า dtype เป็น float32 เพื่อความแม่นยำและการคำนวณที่เหมาะสมใน PyTorch
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        # 2. ตั้งค่า Seed และสุ่มค่าน้ำหนักเริ่มต้น
        torch.manual_seed(self.random_state)
        self.w_ = torch.normal(mean=0.0, std=0.01, size=(X_tensor.shape[1],))
        print(X_tensor.shape[1],)
        self.b_ = torch.tensor(0.0, dtype=torch.float32)

        self.errors_ = []

        for _ in range(self.n_iter):
            # 3. คำนวณค่าเน็ตอินพุตและทำนายผล
            net_output = self.net_input(X_tensor)
            output = torch.where(net_output >= 0.0, 1.0, 0.0)

            # 4. หาค่าความผิดพลาด
            errors = (y_tensor - output)

            # 5. อัปเดตค่าน้ำหนักด้วย Vectorization (ใช้ torch.matmul แทน np.dot)
            self.w_ += self.eta * torch.matmul(X_tensor.T, errors)
            self.b_ += self.eta * errors.sum()

            # 6. บันทึกจำนวนตัวที่ทายผิด (ใช้ .item() เพื่อแปลง Tensor กลับเป็นตัวเลขปกติ)
            self.errors_.append(int(torch.sum(errors != 0).item()))

        return self

    def net_input(self, X):
        """Calculate net input using PyTorch matrix multiplication"""
        return torch.matmul(X, self.w_) + self.b_

    def predict(self, X):
        """Return class label and convert back to NumPy array for compatibility"""
        # รองรับทั้งข้อมูลที่ส่งเข้ามาเป็น NumPy และ Tensor
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)

        predictions = torch.where(self.net_input(X) >= 0.0, 1, 0)
        return predictions.numpy() # แปลงกลับเป็น NumPy เพื่อให้นำไปพล็อตกราฟได้ง่าย


# --- ส่วนทดสอบการทำงาน ---
if __name__ == "__main__":
    # สร้างข้อมูลจำลอง
    X, y = make_classification(n_samples=100, n_features=2, n_redundant=0,
                               n_informative=2, random_state=42, n_clusters_per_class=1)

    # เทรนโมเดลด้วย PyTorch
    classifier = PerceptronGDPyTorch(eta=0.01, n_iter=15, random_state=42)
    classifier.fit(X, y)

    print("จำนวนความผิดพลาดในแต่ละรอบ (Errors):", classifier.errors_)
    print("ค่าน้ำหนักสุดท้าย (Weights):", classifier.w_.numpy())
    print("ค่าไบแอสสุดท้าย (Bias):", classifier.b_.item())

    # ทดสอบทำนายข้อมูลใหม่
    X_test = np.array([[2.5, 3.1], [-1.2, -2.5], [1.5, -5.3], [-2.3, 4.0], [1.0, 2.0], [5.0, -5.0]])
    predictions = classifier.predict(X_test)
    print("ผลการทำนายข้อมูลใหม่:", predictions)

2
จำนวนความผิดพลาดในแต่ละรอบ (Errors): [46, 50, 5, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1]
ค่าน้ำหนักสุดท้าย (Weights): [-0.14799699  0.7333729 ]
ค่าไบแอสสุดท้าย (Bias): 0.40000006556510925
ผลการทำนายข้อมูลใหม่: [1 0 0 1 1 0]


#MutiPerceptronPytorch

In [ ]:
import torch
import torch.nn as nn

class PyTorchMultiNodePerceptron(nn.Module):
    def __init__(self, n_inputs, n_neurons):
        super().__init__()
        # nn.Linear จะสร้าง Weight Matrix ขนาด (n_neurons, n_inputs)
        # และ Bias Vector ขนาด (n_neurons) ให้โดยอัตโนมัติ
        self.linear = nn.Linear(n_inputs, n_neurons)

    def forward(self, x):
        print(f"x:{x}")
        # คำนวณสมการเส้นตรง: net_input = x * W^T + b
        net_input = self.linear(x)

        # Step Function: ถ้าค่า >= 0 ให้เป็น 1, ถ้าไม่ใช่ให้เป็น 0
        # ใช้ torch.where เพื่อจัดการข้อมูลทั้ง Tensor รวดเดียว
        output = torch.where(net_input >= 0.0, torch.tensor(1.0), torch.tensor(0.0))
        return output

# --- ลองทดสอบ ---
# 1. แปลงข้อมูล NumPy เป็น PyTorch Tensor (ต้องระบุ dtype เป็น float32)
X_test = torch.tensor([
    [1.5, 2.0, -1.0],
    [0.5, -1.5, 2.5]
], dtype=torch.float32)

# 2. สร้างโมเดล: รับ 3 ฟีเจอร์ และมี Perceptron 4 ตัว
model = PyTorchMultiNodePerceptron(n_inputs=3, n_neurons=4)

# 3. ทำนายผล (Forward pass)
# ใส่ torch.no_grad() เพื่อบอก PyTorch ว่าไม่ต้องจำการคำนวณสำหรับหา Gradient
# เพราะตอนนี้เราแค่ทำนายผล (Predict) ยังไม่ได้ Train
with torch.no_grad():
    outputs = model(X_test)

print("Shape ของผลลัพธ์:", outputs.shape)
print("ผลลัพธ์ที่ได้:\n", outputs)

x:tensor([[ 1.5000,  2.0000, -1.0000],
        [ 0.5000, -1.5000,  2.5000]])
Shape ของผลลัพธ์: torch.Size([2, 4])
ผลลัพธ์ที่ได้:
 tensor([[1., 1., 1., 1.],
        [0., 0., 1., 0.]])


#Perceptron 1 to 1 layer


In [ ]:
import torch
import torch.nn as nn

class TwoLayerPerceptron(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        # ชั้นที่ 1 (Hidden Layer): รับข้อมูล n_inputs ฟีเจอร์ เข้ามาที่ Perceptron 1 ตัว
        self.layer1 = nn.Linear(n_inputs, 1)

        # ชั้นที่ 2 (Output Layer): รับผลลัพธ์ 1 ค่าจากชั้นแรก เข้ามาที่ Perceptron 1 ตัว
        self.layer2 = nn.Linear(1, 1)

    def forward(self, x):
        # --- การทำงานในชั้นที่ 1 ---
        # 1. คำนวณสมการเส้นตรง
        net_input1 = self.layer1(x)
        # 2. ผ่าน Step Function (ผลลัพธ์จะเป็น 0 หรือ 1)
        out1 = torch.where(net_input1 >= 0.0, torch.tensor(1.0), torch.tensor(0.0))

        # --- การทำงานในชั้นที่ 2 ---
        # 3. เอาผลลัพธ์จากชั้นแรก (out1) มาเข้าสมการเส้นตรงของชั้นที่ 2
        net_input2 = self.layer2(out1)
        # 4. ผ่าน Step Function อีกครั้งเพื่อได้ผลลัพธ์สุดท้าย
        out2 = torch.where(net_input2 >= 0.0, torch.tensor(1.0), torch.tensor(0.0))

        return out2

# --- ลองทดสอบ ---
# จำลองข้อมูล 2 ตัวอย่าง แต่ละตัวมี 3 ฟีเจอร์
X_test = torch.tensor([
    [1.5, 2.0, -1.0],
    [0.5, -1.5, 2.5]
], dtype=torch.float32)

# สร้างโมเดลแบบ 2 ชั้น
model = TwoLayerPerceptron(n_inputs=3)

# ทำนายผล
with torch.no_grad():
    outputs = model(X_test)

print("Shape ของผลลัพธ์:", outputs.shape)
print("ผลลัพธ์ที่ได้:\n", outputs)

Shape ของผลลัพธ์: torch.Size([2, 1])
ผลลัพธ์ที่ได้:
 tensor([[0.],
        [0.]])


#Multi-Layer Perceptron (MLP)

In [ ]:
import torch
import torch.nn as nn

class MultiLayerMultiNodePerceptron(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()
        # ชั้นที่ 1 (Hidden Layer):
        # รับข้อมูลตั้งต้น (n_inputs) และมี Perceptron ขนานกันจำนวน n_hidden ตัว
        self.layer1 = nn.Linear(n_inputs, n_hidden)
        print(f"self.layer1:{self.layer1}")

        # ชั้นที่ 2 (Output Layer):
        # รับผลลัพธ์จากชั้นที่ 1 (n_hidden) และมี Perceptron ขนานกันจำนวน n_outputs ตัว
        self.layer2 = nn.Linear(n_hidden, n_outputs)
        print(f"self.layer2:{self.layer2}")

    def forward(self, x):
        # --- ชั้นที่ 1 (Hidden Layer) ---
        net_input1 = self.layer1(x)
        print(f"netput1:{net_input1}")
        out1 = torch.where(net_input1 >= 0.0, torch.tensor(1.0), torch.tensor(0.0))
        print(f"out1:{out1}")

        # --- ชั้นที่ 2 (Output Layer) ---
        # นำ out1 ซึ่งเป็นผลลัพธ์ของ Perceptron หลายตัวในชั้นแรก มาเป็น Input ให้ชั้นที่ 2
        net_input2 = self.layer2(out1)
        print(f"netput2:{net_input2}")
        out2 = torch.where(net_input2 >= 0.0, torch.tensor(1.0), torch.tensor(0.0))

        return out2

# --- ลองทดสอบ ---
# จำลองข้อมูล 2 ตัวอย่าง แต่ละตัวมี 3 ฟีเจอร์
X_test = torch.tensor([
    [1.5, 2.0, -1.0],
    [0.5, -1.5, 2.5],
    [1.0, 2.0, 3.0],
    [1.5, -2.5, 3.5]
], dtype=torch.float32)

# สร้างโมเดล:
# - Input: 3 ฟีเจอร์
# - ชั้นที่ 1 (Hidden): มี Perceptron 4 ตัว
# - ชั้นที่ 2 (Output): มี Perceptron 2 ตัว (เช่น ทายผล 2 คลาส หรือทาย 2 คำตอบพร้อมกัน)
model = MultiLayerMultiNodePerceptron(n_inputs=3, n_hidden=4, n_outputs=2)

# ทำนายผล
with torch.no_grad():
    outputs = model(X_test)

print("Shape ของผลลัพธ์:", outputs.shape)
print("ผลลัพธ์ที่ได้:\n", outputs)

self.layer1:Linear(in_features=3, out_features=4, bias=True)
self.layer2:Linear(in_features=4, out_features=2, bias=True)
netput1:tensor([[-0.9302,  0.4958,  1.2034,  0.0921],
        [ 0.6385, -2.4754, -0.2348,  1.0530],
        [-0.6307, -1.1808,  0.2306,  1.7796],
        [ 0.8772, -4.0575, -0.0726,  1.7863]])
out1:tensor([[0., 1., 1., 1.],
        [1., 0., 0., 1.],
        [0., 0., 1., 1.],
        [1., 0., 0., 1.]])
netput2:tensor([[ 0.0219, -0.1546],
        [-0.5031, -0.2093],
        [-0.4245, -0.2440],
        [-0.5031, -0.2093]])
Shape ของผลลัพธ์: torch.Size([4, 2])
ผลลัพธ์ที่ได้:
 tensor([[1., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]])


#XOR Gate

In [ ]:
import torch
import torch.nn as nn

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ใช้ nn.Sequential เพื่อร้อยเรียงเลเยอร์ให้โค้ดอ่านง่ายขึ้น
        self.model = nn.Sequential(
            nn.Linear(2, 4),      # ชั้นที่ 1: รับ 2 ส่งออก 4
            nn.ReLU(),            # ฟังก์ชันกระตุ้น (ดัดเส้นตรงให้โค้งงอได้)
            nn.Linear(4, 1),      # ชั้นที่ 2: รับ 4 ส่งออก 1
            nn.Sigmoid()          # แปลงผลลัพธ์สุดท้ายให้อยู่ในช่วง 0-1
        )

    def forward(self, x):
        return self.model(x)

# การใช้งาน
x_data = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
model_level1 = XORNet()
print("Level 1 Output Shape:", model_level1(x_data).shape) # (4, 1)
print("output:",model_level1(x_data))

Level 1 Output Shape: torch.Size([4, 1])
output: tensor([[0.4499],
        [0.4707],
        [0.4485],
        [0.4728]], grad_fn=<SigmoidBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. เตรียมข้อมูล Input และ Output (เฉลยของตรรกะ XOR)
x_data = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_data = torch.tensor([[0.0], [1.0], [1.0], [0.0]]) # ถ้าเหมือนกัน=0, ต่างกัน=1

# 2. สร้างโมเดล (ใช้โค้ดเดิม)
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

model = XORNet()

# 3. กำหนดตัววัดความผิดพลาด (Loss) และตัวปรับน้ำหนัก (Optimizer)
# ใช้ Binary Cross Entropy Loss สำหรับปัญหาที่ตอบ 0 หรือ 1
criterion = nn.BCELoss()
# ใช้ Adam Optimizer ช่วยคำนวณและปรับ Weight อัตโนมัติ (กำหนด Learning Rate = 0.01)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. เริ่ม Training Loop
epochs = 1000# ให้โมเดลฝึกซ้ำๆ 1,000 รอบ

print("เริ่มการ Train...")
for epoch in range(epochs):
    # สเต็ปที่ 1: Forward Pass - ป้อนข้อมูลให้โมเดลลองทายผล
    predictions = model(x_data)

    # สเต็ปที่ 2: Compute Loss - เทียบผลทายกับเฉลย ว่ามีความผิดพลาดอยู่เท่าไหร่
    loss = criterion(predictions, y_data)

    # สเต็ปที่ 3: Zero Gradients - ล้างค่าความชันเก่าของรอบที่แล้วทิ้ง
    optimizer.zero_grad()

    # สเต็ปที่ 4: Backward Pass - คำนวณความชัน (Backpropagation)
    loss.backward()

    # สเต็ปที่ 5: Update Weights - ปรับค่า Weight ในโมเดลให้ดีขึ้น
    optimizer.step()

    # พิมพ์ผลลัพธ์ทุกๆ 200 รอบ ให้เห็นว่า Loss ลดลงเรื่อยๆ
    if (epoch + 1) % 200 == 0:
        print(f"Epoch: {epoch+1:4d} | Loss: {loss.item():.4f}")

# 5. ทดสอบผลลัพธ์หลัง Train เสร็จ
print("\n--- ผลลัพธ์หลังจาก Train ---")
with torch.no_grad(): # ปิดการคำนวณ Gradient เพราะตอนนี้แค่เรียกใช้งาน
    final_outputs = model(x_data)
    final_preds = (final_outputs >= 0.5).int() # ถ้าเกิน 0.5 ให้ปัดเป็น 1

    for i in range(len(x_data)):
        print(f"Input: {x_data[i].tolist()} -> ทายผล: {final_preds[i].item()} (ความน่าจะเป็น: {final_outputs[i].item():.4f})")

เริ่มการ Train...
Epoch:  200 | Loss: 0.2698
Epoch:  400 | Loss: 0.0185
Epoch:  600 | Loss: 0.0056
Epoch:  800 | Loss: 0.0030
Epoch: 1000 | Loss: 0.0018

--- ผลลัพธ์หลังจาก Train ---
Input: [0.0, 0.0] -> ทายผล: 0 (ความน่าจะเป็น: 0.0026)
Input: [0.0, 1.0] -> ทายผล: 1 (ความน่าจะเป็น: 0.9983)
Input: [1.0, 0.0] -> ทายผล: 1 (ความน่าจะเป็น: 0.9981)
Input: [1.0, 1.0] -> ทายผล: 0 (ความน่าจะเป็น: 0.0010)


#Project 2 MNIST


In [ ]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader


image_path = './'
transform = transforms.Compose([transforms.ToTensor()])

mnist_train_dataset = torchvision.datasets.MNIST(root=image_path,
                                           train=True,
                                           transform=transform,
                                           download=True)
mnist_test_dataset = torchvision.datasets.MNIST(root=image_path,
                                           train=False,
                                           transform=transform,
                                           download=False)

batch_size = 64
torch.manual_seed(1)
train_dl = DataLoader(mnist_train_dataset, batch_size, shuffle=True)

In [ ]:
hidden_units = [32, 16]
image_size = mnist_train_dataset[0][0].shape
input_size = image_size[0] * image_size[1] * image_size[2]

all_layers = [nn.Flatten()]
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 10))
model = nn.Sequential(*all_layers)

model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=32, bias=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Linear(in_features=16, out_features=10, bias=True)
)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.manual_seed(1)
num_epochs = 20
for epoch in range(num_epochs):
    accuracy_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
        accuracy_hist_train += is_correct.sum()
    accuracy_hist_train /= len(train_dl.dataset)
    print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}')

Epoch 0  Accuracy 0.8531
Epoch 1  Accuracy 0.9287
Epoch 2  Accuracy 0.9413
Epoch 3  Accuracy 0.9506
Epoch 4  Accuracy 0.9558
Epoch 5  Accuracy 0.9592
Epoch 6  Accuracy 0.9627
Epoch 7  Accuracy 0.9649
Epoch 8  Accuracy 0.9673
Epoch 9  Accuracy 0.9690
Epoch 10  Accuracy 0.9711
Epoch 11  Accuracy 0.9729
Epoch 12  Accuracy 0.9737
Epoch 13  Accuracy 0.9747
Epoch 14  Accuracy 0.9766
Epoch 15  Accuracy 0.9778
Epoch 16  Accuracy 0.9780
Epoch 17  Accuracy 0.9798
Epoch 18  Accuracy 0.9806
Epoch 19  Accuracy 0.9816


In [ ]:
pred = model(mnist_test_dataset.data / 255.)
is_correct = (torch.argmax(pred, dim=1) == mnist_test_dataset.targets).float()
print(f'Test accuracy: {is_correct.mean():.4f}')

Test accuracy: 0.9645


#Gradient Descent MNIST

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ==========================================
# 1. Device
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ==========================================
# 2. MNIST Dataset
# ==========================================

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=False,
    transform=transform
)


# ==========================================
# 3. DataLoader
# ==========================================

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ==========================================
# 4. Model
# ==========================================

class MNISTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(28 * 28, 10)

    def forward(self, x):

        # [batch, 1, 28, 28]
        x = x.view(x.size(0), -1)

        # [batch, 784]
        x = self.linear(x)

        # [batch, 10]
        return x


model = MNISTModel().to(device)


# ==========================================
# 5. Loss Function
# ==========================================

loss_function = nn.CrossEntropyLoss()


# ==========================================
# 6. Learning Rate
# ==========================================

learning_rate = 0.01


# ==========================================
# 7. Training
# ==========================================

epochs = 20

for epoch in range(epochs):

    total_loss = 0

    for images, labels in train_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)


        # ----------------------------------
        # Forward
        # ----------------------------------

        predictions = model(images)


        # ----------------------------------
        # Calculate Loss
        # ----------------------------------

        loss = loss_function(
            predictions,
            labels
        )


        # ----------------------------------
        # Gradient Descent
        # ----------------------------------

        # Clear old gradients
        model.zero_grad()

        # Calculate gradients
        loss.backward()


        # Update parameters
        with torch.no_grad():

            for parameter in model.parameters():

                parameter -= learning_rate * parameter.grad


        total_loss += loss.item()


    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {average_loss:.4f}"
    )


# ==========================================
# 8. Test
# ==========================================

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        predictions = model(images)

        predicted_class = predictions.argmax(dim=1)

        total += labels.size(0)

        correct += (
            predicted_class == labels
        ).sum().item()


accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Device: cpu
Epoch [1/20] Loss: 0.9817
Epoch [2/20] Loss: 0.5524
Epoch [3/20] Loss: 0.4725
Epoch [4/20] Loss: 0.4334
Epoch [5/20] Loss: 0.4094
Epoch [6/20] Loss: 0.3925
Epoch [7/20] Loss: 0.3798
Epoch [8/20] Loss: 0.3699
Epoch [9/20] Loss: 0.3618
Epoch [10/20] Loss: 0.3551
Epoch [11/20] Loss: 0.3492
Epoch [12/20] Loss: 0.3441
Epoch [13/20] Loss: 0.3396
Epoch [14/20] Loss: 0.3357
Epoch [15/20] Loss: 0.3323
Epoch [16/20] Loss: 0.3290
Epoch [17/20] Loss: 0.3260
Epoch [18/20] Loss: 0.3236
Epoch [19/20] Loss: 0.3213
Epoch [20/20] Loss: 0.3190
Test Accuracy: 91.58%


#CNN